In [2]:
import pandas as pd
import random
import re

In [3]:
gemini_paraphrase_df = pd.read_csv('../data/augmented/gemini_paraphrase_1nf.csv').drop(columns=['text'], inplace=False)
ar2en2ar_df = pd.read_csv('../data/augmented/ar2en2ar_auglmentation.csv')
processed_raw_df = pd.read_csv('../data/raw/processed_original.csv')

In [4]:
gemini_paraphrase_df = gemini_paraphrase_df.rename(columns={"paraphrased_text": "text"})
gemini_paraphrase_df.head()

,intent,text
0,check_balance,ما هو رصيد البطاقة مسبقة الدفع
1,check_balance,كم يبلغ رصيد بطاقة الدفع المسبق
2,check_balance,استعلم عن رصيد البطاقة المسبقة
3,check_balance,أظهر رصيد البطاقة مسبقة الدفع
4,check_balance,ما هي القيمة المتوفرة في بطاقتك المسبقة


In [5]:
combin_df = pd.concat(
    [gemini_paraphrase_df, ar2en2ar_df, processed_raw_df],
    ignore_index=True
)

In [6]:
len(combin_df)

841

In [7]:
filtered_df_NER = combin_df[combin_df["intent"].isin(["p2p_transfer", "pay_bill"])]

# Optional: reset index
filtered_df_NER = filtered_df_NER.reset_index(drop=True)
filtered_df_NER.to_csv("../data/ner/filtered_for_NER.csv", index=False)
len(filtered_df_NER)

420

In [8]:
combin_df['intent'].value_counts()

intent
unknown          211
check_balance    210
pay_bill         210
p2p_transfer     210
Name: count, dtype: int64

In [9]:
def inject_typo(text, typo_prob=0.3):
    # If random condition not met, return original text
    if random.random() > typo_prob:
        return text
    
    text = list(text)
    
    if len(text) < 2:
        return "".join(text)
    
    typo_type = random.choice(["delete", "swap", "replace", "repeat"])
    idx = random.randint(0, len(text)-1)
    
    if typo_type == "delete":
        del text[idx]
    
    elif typo_type == "swap" and idx < len(text)-1:
        text[idx], text[idx+1] = text[idx+1], text[idx]
    
    elif typo_type == "replace":
        arabic_letters = "ابتثجحخدذرزسشصضطظعغفقكلمنهوي"
        text[idx] = random.choice(arabic_letters)
    
    elif typo_type == "repeat":
        text.insert(idx, text[idx])
    
    return "".join(text)


In [10]:
def augment_with_typo(df, text_col="text", n_samples=10, typo_prob=0.5):
    """
    df: original dataframe
    text_col: name of text column
    n_samples: number of new augmented samples to generate
    typo_prob: probability of injecting typo
    """
    
    augmented_rows = []
    
    for _ in range(n_samples):
        # randomly select a row
        row = df.sample(1).iloc[0]
        
        new_row = row.copy()
        new_row[text_col] = inject_typo(row[text_col], typo_prob)
        
        augmented_rows.append(new_row)
    
    df_aug = pd.DataFrame(augmented_rows)
    
    # combine original + augmented
    df_final = pd.concat([df, df_aug], ignore_index=True)
    
    return df_final


In [11]:
def normalize_arabic(text):
    # توحيد الحروف المتشابهة
    text = text.replace("إ", "ا").replace("أ", "ا").replace("آ", "ا")
    text = text.replace("ى", "ي").replace("ة", "ه")
    
    # إزالة الرموز غير الحروف والأرقام
    text = re.sub(r"[^ء-ي0-9\s]", "", text)
    
    # إزالة المسافات الزائدة
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [12]:
df_final = augment_with_typo(
    combin_df,
    text_col="text",
    n_samples=1000,      # number of new typo samples
    typo_prob=0.8
)

len(df_final)

1841

In [13]:
df_final["intent"].value_counts()

intent
unknown          475
p2p_transfer     470
check_balance    449
pay_bill         447
Name: count, dtype: int64

In [14]:
# Shuffle the entire DataFrame
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

In [15]:
df_final["text"] = df_final["text"].apply(normalize_arabic)
df_final

,intent,text
0,unknown,ادخل الي الرسائل
1,p2p_transfer,انقل 22 دينارا لرنا في اللحال
2,check_balance,كشف الرصيد المتوفر حاليا
3,pay_bill,سدد فاتوره اظمياه
4,unknown,شغلل البلوتوث
...,...,...
1836,p2p_transfer,قم بتحويل 45 دينارا الي يوسف
1837,p2p_transfer,قم بارسال 50 دينارا لهبه
1838,p2p_transfer,حول 5ث دينار الي مريم
1839,unknown,اوجد اكان الجامعه


In [16]:
print("duplicated:", df_final["text"].duplicated().sum())
df_final.drop_duplicates(inplace=True)

duplicated: 294


In [17]:
len(df_final)

1547

In [18]:
df_final.to_csv("../data/augmented/intent_preparation_and_augmentation_combination.csv", index=False)